# Vihangani G.S.A. IT22081452 — K-Nearest Neighbors(KNN)

**Dataset:** Titanic Survival  
**Algorithm:** NK-Nearest Neighbors(KNN) 
**Goal:** Predict whether a passenger survived (1) or not (0)

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

# Make plots look nice
sns.set_theme(style="whitegrid")
%matplotlib inline


In [2]:
# ── Load Dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv("Titanic_Dataset.csv")
print("Shape:", df.shape)
df.head()


Shape: (1309, 14)


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.00,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.92,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.00,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.00,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.00,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [3]:
# ── Explore the Data ─────────────────────────────────────────────────────────
print("=== Column Info ===")
df.info()
print()
print("=== Missing Values ===")
print(df.isnull().sum())
print()
print("=== Survival Count ===")
print(df["survived"].value_counts())


=== Column Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   pclass     1309 non-null   int64  
 1   survived   1309 non-null   int64  
 2   name       1309 non-null   object 
 3   sex        1309 non-null   object 
 4   age        1046 non-null   float64
 5   sibsp      1309 non-null   int64  
 6   parch      1309 non-null   int64  
 7   ticket     1309 non-null   object 
 8   fare       1308 non-null   float64
 9   cabin      295 non-null    object 
 10  embarked   1307 non-null   object 
 11  boat       486 non-null    object 
 12  body       121 non-null    float64
 13  home.dest  745 non-null    object 
dtypes: float64(3), int64(4), object(7)
memory usage: 143.3+ KB

=== Missing Values ===
pclass          0
survived        0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
f

In [ ]:
# ── Clean the Data ───────────────────────────────────────────────────────────

# 1. Drop columns that are not useful or would cause data leakage
df.drop(columns=["name", "ticket", "cabin", "boat", "body", "home.dest"],
        inplace=True)

# 2. Fill missing values (using = assignment instead of inplace=True)
df["age"]      = df["age"].fillna(df["age"].median())
df["fare"]     = df["fare"].fillna(df["fare"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

# 3. Convert Sex to numbers: male=0, female=1
df["sex"] = df["sex"].map({"male": 0, "female": 1})

# 4. Convert Embarked to numbers: S=0, C=1, Q=2
df["embarked"] = df["embarked"].map({"S": 0, "C": 1, "Q": 2})

print("Cleaned dataset shape:", df.shape)
print("Any missing values left?", df.isnull().sum().any())
df.head()


In [ ]:
# ── Split into Features (X) and Target (y) ───────────────────────────────────

# X = input columns (what the model learns FROM)
X = df.drop(columns=["survived"])

# y = target column (what the model is PREDICTING)
y = df["survived"]

# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")


In [ ]:
# ── Scale the Features (KNN needs this!) ────────────────────────────────────
# KNN measures distances between data points.
# If one column has very large numbers (e.g. fare = 500)
# and another has small numbers (e.g. pclass = 1,2,3),
# the large numbers will dominate. Scaling fixes this.

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling done!")


In [ ]:
# ── K-Nearest Neighbors Classifier ───────────────────────────────────────────
from sklearn.neighbors import KNeighborsClassifier

# Start with k=5 (5 nearest neighbors)
model = KNeighborsClassifier(n_neighbors=5)

# Train on SCALED data
model.fit(X_train_scaled, y_train)

print("KNN model trained successfully!")
